In [1]:
import sys
sys.path.append("..")
from src.inference import predict_ticket

In [2]:
cases = [
    ("Server outage",
     "Our production servers have been completely down for two hours. "
     "All customers are affected and we cannot process any transactions."),

    ("Invoice question",
     "I noticed a charge on my monthly invoice that I do not recognize. "
     "Could you please clarify what this line item refers to?"),

    ("Feature request",
     "It would be nice if the dashboard allowed exporting reports to Excel. "
     "Not urgent at all, just a suggestion for a future release."),

    ("Cannot log in",
     "I have been unable to access my account since yesterday. "
     "The password reset email never arrives and I have tried three times."),
]

In [3]:
for subject, body in cases:
    r = predict_ticket(subject, body)
    print(f"\n{subject}")
    print(f"  -> {r['queue']} ({r['queue_confidence']:.2f})")
    print(f"  -> priority {r['priority']} ({r['priority_confidence']:.2f})")


Server outage
  -> Technical Support (0.24)
  -> priority high (0.68)

Invoice question
  -> Billing and Payments (0.66)
  -> priority medium (0.52)

Feature request
  -> Technical Support (0.38)
  -> priority medium (0.50)

Cannot log in
  -> Billing and Payments (0.56)
  -> priority high (0.58)


In [4]:
print(predict_ticket("help", "urgent"))

{'error': 'text_too_short', 'message': 'Ticket text is too short to classify reliably.'}


In [5]:
import time
t0 = time.perf_counter()
for _ in range(100):
    predict_ticket("Test ticket", "The application crashes when I open the settings page.")
elapsed = (time.perf_counter() - t0) / 100
print(f"Average inference time: {elapsed*1000:.2f} ms")

Average inference time: 8.58 ms


In [6]:
from pathlib import Path
for f in sorted(Path("../models").glob("*.joblib")):
    print(f"{f.name:30s} {f.stat().st_size/1024:8.1f} KB")

priority_classifier.joblib       2348.8 KB
queue_classifier.joblib          7821.8 KB
tfidf_vectorizer.joblib           827.9 KB


In [8]:
import joblib, numpy as np, pandas as pd
from pathlib import Path

PROC_DIR = Path("../data/processed")
d = joblib.load(PROC_DIR / "splits.joblib")
Xte, y_test_q = d["Xte"], d["y_test_q"]

clf = joblib.load("../models/queue_classifier.joblib")
proba = clf.predict_proba(Xte)
pred = clf.classes_[proba.argmax(axis=1)]
conf = proba.max(axis=1)
correct = (pred == y_test_q.values)

rows = []
for t in [0.0, 0.3, 0.4, 0.5, 0.6, 0.7]:
    keep = conf >= t
    rows.append({
        "threshold": t,
        "auto_routed_%": round(100 * keep.mean(), 1),
        "accuracy_when_routed": round(correct[keep].mean(), 3) if keep.sum() else None,
        "sent_to_triage_%": round(100 * (~keep).mean(), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

 threshold  auto_routed_%  accuracy_when_routed  sent_to_triage_%
       0.0          100.0                 0.556               0.0
       0.3           87.1                 0.590              12.9
       0.4           56.2                 0.694              43.8
       0.5           31.0                 0.808              69.0
       0.6           16.2                 0.889              83.8
       0.7            7.1                 0.952              92.9


In [9]:
clf_p = joblib.load("../models/priority_classifier.joblib")
y_test_p = d["y_test_p"]

proba_p = clf_p.predict_proba(Xte)
pred_p = clf_p.classes_[proba_p.argmax(axis=1)]
conf_p = proba_p.max(axis=1)
correct_p = (pred_p == y_test_p.values)

rows = []
for t in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8]:
    keep = conf_p >= t
    rows.append({
        "threshold": t,
        "auto_%": round(100 * keep.mean(), 1),
        "accuracy": round(correct_p[keep].mean(), 3) if keep.sum() else None,
        "triage_%": round(100 * (~keep).mean(), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

 threshold  auto_%  accuracy  triage_%
       0.0   100.0     0.602       0.0
       0.4    92.4     0.624       7.6
       0.5    55.7     0.700      44.3
       0.6    22.2     0.783      77.8
       0.7     4.3     0.863      95.7
       0.8     0.4     0.889      99.6


In [10]:
import numpy as np
high_mask = (y_test_p.values == "high")
print(f"Mean confidence on true HIGH tickets:  {conf_p[high_mask].mean():.3f}")
print(f"Mean confidence on all other tickets:  {conf_p[~high_mask].mean():.3f}")

missed = high_mask & (pred_p != "high")
print(f"\nMissed critical tickets: {missed.sum()} of {high_mask.sum()}")
print(f"Their mean confidence:   {conf_p[missed].mean():.3f}")

Mean confidence on true HIGH tickets:  0.537
Mean confidence on all other tickets:  0.519

Missed critical tickets: 319 of 906
Their mean confidence:   0.505


In [11]:
import re, pandas as pd

CRITICAL_TERMS = [
    "outage", "down", "breach", "unauthorized", "urgent", "critical",
    "cannot access", "data loss", "security incident", "not working",
    "emergency", "immediately", "asap", "blocked", "failure",
]

df_clean = pd.read_csv(PROC_DIR / "tickets_clean_en.csv")
from sklearn.model_selection import train_test_split
_, te_df = train_test_split(df_clean, test_size=0.2, random_state=42, stratify=df_clean["queue"])

pattern = re.compile("|".join(re.escape(t) for t in CRITICAL_TERMS))
has_term = te_df["text"].str.contains(pattern, regex=True).values

true_high = (te_df["priority"].values == "high")

print(f"Tickets matching a critical term: {has_term.sum()} of {len(te_df)} "
      f"({100*has_term.mean():.1f}%)")
print(f"  of those, actually high:        {true_high[has_term].mean():.3f}")
print(f"  base rate of high overall:      {true_high.mean():.3f}")

Tickets matching a critical term: 555 of 2381 (23.3%)
  of those, actually high:        0.506
  base rate of high overall:      0.381


In [12]:
import numpy as np
missed_mask = true_high & (pred_p != "high")
print(f"\nMissed criticals: {missed_mask.sum()}")
print(f"  caught by keyword rule: {(missed_mask & has_term).sum()} "
      f"({100*(missed_mask & has_term).sum()/missed_mask.sum():.1f}%)")

# cost of the rule: non-high tickets it would wrongly escalate
false_esc = (~true_high) & has_term & (pred_p != "high")
print(f"  false escalations:      {false_esc.sum()} "
      f"({100*false_esc.sum()/(~true_high).sum():.1f}% of non-high tickets)")


Missed criticals: 319
  caught by keyword rule: 76 (23.8%)
  false escalations:      184 (12.5% of non-high tickets)


In [13]:
for term in CRITICAL_TERMS:
    m = te_df["text"].str.contains(re.escape(term), regex=True).values
    if m.sum() >= 20:
        print(f"{term:20s} n={m.sum():4d}  P(high)={true_high[m].mean():.3f}")

outage               n=  48  P(high)=0.750
down                 n=  55  P(high)=0.618
breach               n= 192  P(high)=0.401
unauthorized         n= 129  P(high)=0.442
urgent               n= 144  P(high)=0.535
critical             n=  71  P(high)=0.634
security incident    n=  23  P(high)=0.391
failure              n= 103  P(high)=0.505
